# Applicant name harmonisation – quantum computing applicants

**Strategy.** We reuse the general pipeline tested on the gold standard (`harmonisation_gold.ipynb`): **clean → brand key → fuzzy**, with the same functions. Only three sector steps are added around it:
1. **Data preparation**: one row per family × applicant; the raw name is `han_name` (OECD harmonised name) or `psn_name` when missing.
2. **Chinese/Japanese names** of big brands are translated (e.g. 国际商业机器 → IBM).
3. **Applicant type**: only companies, universities and government / non-profit bodies are kept; the pipeline runs separately for each type, so a university is never merged with a company.

The list of generic words is extended with sector words (QUANTUM, COMPUTING…) and cities that are not brands (OXFORD, TOKYO…).

Output: one row per family × organisation with `group_id`, `group_name`, `entity_type` and `weight` (1 / number of organisations of the family, for fractional counting).

In [21]:
import re
import unicodedata
import pandas as pd
import networkx as nx
from rapidfuzz.fuzz import ratio

INPUT_CSV = "../qc_applicants_PROD.csv"
OUTPUT_CSV = "qc_applicants_harmonised.csv"
COLS = ["docdb_family_id", "psn_id", "psn_name", "psn_sector", "han_name"]
FUZZY_MIN = 90
MIN_KEY_LEN = 6

## 1. Data preparation
One row per family × applicant (the same applicant appears once per application of the family).

In [22]:
df = pd.read_csv(INPUT_CSV, usecols=COLS).drop_duplicates(["docdb_family_id", "psn_id"])
df["raw_name"] = df["han_name"].fillna(df["psn_name"])

## 2. Translation of Chinese / Japanese brand names

In [23]:
CJK_BRAND_MAP = {
    "国际商业机器": "IBM",
    "谷歌": "GOOGLE",
    "微软技术许可": "MICROSOFT TECH LICENSING",
    "微软": "MICROSOFT",
    "爱奥尼克": "IONQ",
    "霍尼韦尔国际": "HONEYWELL INTERNATIONAL",
    "英特尔": "INTEL",
    "1QB信息技术": "1QB INFORMATION TECH",
    "ID量子技术": "ID QUANTIQUE",
    "亚马逊科技": "AMAZON TECH",
    "亚马逊技术": "AMAZON TECH",
    "硅量子计算": "SILICON QUANTUM COMPUTING",
    "IQM芬兰": "IQM FINLAND",
    "普赛昆腾": "PSIQUANTUM",
    "赞纳杜量子技术": "XANADU QUANTUM TECH",
    "剑桥量子计算": "CAMBRIDGE QUANTUM COMPUTING",
    "多元宇宙计算": "MULTIVERSE COMPUTING",
    "泰勒斯": "THALES",
    "量子比特移动与存储": "QUANTUM MOTION TECH",
    "三菱电机": "MITSUBISHI ELECTRIC",
}


def map_cjk(name):
    for cjk, latin in CJK_BRAND_MAP.items():
        name = str(name).replace(cjk, f" {latin} ")
    return name


df["raw_name"] = df["raw_name"].map(map_cjk)

## 3. Applicant type
PATSTAT `psn_sector` when known, otherwise rules on the name. Individuals and uncertain names are dropped.

In [24]:
LEGAL_SEC = [
    "INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY",
    "GMBH", "AG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS",
    "ASA", "PTY", "PLC", "KK", "LLP", "LP", "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK",
    "ANONIM SIRKETI", "OBSHCHESTVO"
]
LEGAL_CJK_SEC = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]

UNIV_SEC = [
    "UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL",
    "INSTITUTE OF TECHNOLOGY", "GIJUKU", "TRUSTEES", "EDUCATIONAL FUND",
    "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"
]

GOV_SEC = [
    "INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY",
    "CENTER", "CENTRE", "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG",
    "FRAUNHOFER", "FROUNHAV", "PLANCK", "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES",
    "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY", "COUNCIL", "CONSEJO",
    "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
    "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"
]

FIRM_SEC = [
    "TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK",
    "ELECTRONIC", "SEMICONDUCT", "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL",
    "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE", "DIGITAL", "DATA", "INTELLIGEN", "ENERGY",
    "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION", "BLOCKCHAIN",
    "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"
]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL_SEC) + r")\b|" + "|".join(LEGAL_CJK_SEC))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV_SEC, GOV_SEC, FIRM_SEC))

WORD_PAT = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"
PERSON_COMMA = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{0,3}} ?, ?{WORD_PAT}")
PERSON_WORDS = re.compile(rf"^{WORD_PAT}( {WORD_PAT}){{1,3}}$")


def _norm_for_sector(name: str) -> str:
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")


def impute_psn_sector(psn_name: str, psn_sector: str) -> str:
    "Keep the PATSTAT sector if known, otherwise guess it from the name."
    s = str(psn_sector).strip().upper() if pd.notna(psn_sector) else ""
    if s and s != "UNKNOWN":
        if "UNIVERSITY" in s:             return "UNIVERSITY"
        if "GOV" in s or "HOSPITAL" in s: return "GOV NON-PROFIT"
        if "COMPANY" in s:                return "COMPANY"
        if "INDIVIDUAL" in s:             return "INDIVIDUAL"
        return s

    n = _norm_for_sector(psn_name)
    if LEGAL_RE.search(n): return "COMPANY"
    if UNIV_RE.search(n):  return "UNIVERSITY"
    if GOV_RE.search(n):   return "GOV NON-PROFIT"
    if FIRM_RE.search(n):  return "COMPANY"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:
        return "INDIVIDUAL"
    if len(n.split()) == 1:
        return "COMPANY"
    return "UNCERTAIN"

df = df[df["psn_name"].str.strip() != "-NOT AVAILABLE-"]
df["sector"] = [impute_psn_sector(n, s) for n, s in zip(df["psn_name"], df["psn_sector"])]
print(df.groupby("sector").agg(applicants=("psn_id", "nunique"), families=("docdb_family_id", "nunique")))
print("Uncertain (dropped):", df.loc[df["sector"] == "UNCERTAIN", "psn_name"].unique())

TYPES = {"COMPANY": "company", "UNIVERSITY": "university", "GOV NON-PROFIT": "gov / non-profit"}
df = df[df["sector"].isin(TYPES)].copy()
df["entity_type"] = df["sector"].map(TYPES)

                applicants  families
sector                              
COMPANY               2599      8976
GOV NON-PROFIT         313       877
INDIVIDUAL            2287      1181
UNCERTAIN                3         3
UNIVERSITY             855      2989
Uncertain (dropped): ['DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C.A.R'
 "EHNSEHRM (EHNSTITJU NAS'ONAL' DE LJA SANTE EH DE LJA RESHERSH MEDIKAL')"
 'PEARL COHEN ZEDEK LATZER BARATZ']


## 4. Pipeline functions
Same code as the gold standard notebook.

In [26]:
FORMS = [
    "INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION",
    "CO", "COMPANY", "PLC", "GMBH", "MBH", "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL",
    "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE", "KK",
    "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA", "LIMITEE", "LTEE",
    "AKTIENGESELLSCHAFT", "SOCIETE ANONYME", "SOCIEDAD ANONIMA", "SOCIETA PER AZIONI"
]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {
    "UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV",
    "UNIVERSIDAD": "UNIV", "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST",
    "INSTITUTO": "INST", "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
    "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB", "CENTRE": "CENTER",
    "SCIENCES": "SCIENCE", "大學": "大学"
}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}


def clean(name):
    if pd.isna(name):
        return ""
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "ɐ")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)   # keep the Latin part of CJK + Latin names
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"^C/O\s+", "", s).strip()
    s = re.sub(r"(?:,|\s)\s*\bC/O\b.*$", "", s)
    s = re.sub(r"\(\s*A\s+[A-Z ]+\b(?:CORP|CORPORATION|COMPANY|LIMITED|LTD|INC)\b[^)]*\)", " ", s)
    s = s.replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s

In [27]:
def brand_key(name, generic):
    "First non-generic word; a one-letter word is joined to the next (D WAVE); whole name if all words are generic."
    words = name.split()
    for i, w in enumerate(words):
        if len(w) == 1 and i + 1 < len(words):
            return w + " " + words[i + 1]
        if w not in generic and len(w) >= 2 and not w.isdigit():
            return w
    return name


def fuzzy_merge(keys):
    "Map each key to the most frequent key among the almost identical ones."
    counts = keys.value_counts()
    long_keys = sorted(k for k in counts.index if len(k) >= MIN_KEY_LEN)
    g = nx.Graph()
    g.add_nodes_from(counts.index)
    g.add_edges_from((a, b) for i, a in enumerate(long_keys) for b in long_keys[i + 1:] if ratio(a, b) >= FUZZY_MIN)
    return {k: min(c, key=lambda x: (-counts[x], x)) for c in nx.connected_components(g) for k in c}


def harmonise(names, generic):
    keys = names.map(clean).map(lambda c: brand_key(c, generic))
    return keys.map(fuzzy_merge(keys))


def make_ids(labels):
    "HARM_00001 = largest group."
    order = labels.value_counts().sort_index().sort_values(ascending=False, kind="stable").index
    return labels.map({lab: f"HARM_{i:05d}" for i, lab in enumerate(order, 1)})


def make_names(names, ids):
    "Most frequent original name of each group (ties: shortest)."
    d = pd.DataFrame({"id": ids, "name": names}).value_counts().reset_index(name="n")
    d["len"] = d["name"].str.len()
    best = d.sort_values(["id", "n", "len", "name"], ascending=[True, False, True, True]).drop_duplicates("id")
    return ids.map(best.set_index("id")["name"])

## 5. Generic words
Same list as the gold standard notebook, extended with sector words, cities that are not brands and HAN abbreviations.

In [28]:
INSTITUTIONS = set("""
UNIV INST INSTITUTE COLLEGE ACADEMY SCHOOL HOCHSCHULE ECOLE POLITECNICA POLYTECHNICAL POLYTECHNIC
UNIVERSITETT UNIVERSITAET UNIVERSITAT UNIVERSITE UNIVERSIDAD UNIVERSITA UNIVERSITET TECHNISCHE VOCATIONAL
TECH SCIENCE RESEARCH LAB CENTER GROUP CORP COMPANY COLTD HOLDINGS INTERNATIONAL NATIONAL STATE PUBLIC
ROYAL NORMAL GENERAL CENTRAL FEDERAL LIMITED RESPONSIBILITY KORPERSCHAFT OFFENTLICHEN RECHTS BOARD
REGENTS TRUSTEES FOUNDATION ASSOCIATION PRESIDENT FELLOWS HOSPITAL DEPARTMENT MINISTRY GOVERNMENT AGENCY
SECRETARY DIRECTOR COMMITTEE COUNCIL INSTITUTION ADMINISTRATIVE INDEPENDENT DEGLI STUDI NORMALE SUPERIEURE
POSTS COOPERATION ACADEMIC ACADEMIA FEDERALNOE GOSUDARSTVENNOE BYUDZHETNOE OBRAZOVATELNOE UCHREZHDENIE
VYSSHEGO OBRAZOVANIYA GESELLSCHAFT FORDERUNG ANGEWANDTEN FORSCHUNG WISSENSCHAFTEN EV
""".split())

PLACES = set("""
CHINA CHINESE JAPAN KOREA EUROPE EUROPEAN AMERICA AMERICAN UNITED STATES KINGDOM UK US DEUTSCHLAND
GERMANY FRANCE AUSTRALIA BEIJING SHANGHAI SHENZHEN GUANGDONG ANHUI JIANGSU ZHEJIANG SICHUAN HUBEI HUNAN
HENAN HEBEI FUJIAN SHANDONG SHANXI SHAANXI LIAONING JILIN HEILONGJIANG YUNNAN GUIZHOU GANSU QINGHAI
HAINAN NINGXIA XINJIANG XIZANG GUANGXI INNER MONGOLIA HEFEI NANJING SUZHOU HANGZHOU WUHAN CHENGDU
CHONGQING DALIAN HARBIN TIANJIN XIAN CHANGCHUN SHENYANG ZHENGZHOU CHANGSHA FUZHOU XIAMEN JINAN KUNMING
TAIYUAN DONGGUAN QINGDAO WUXI NORTH SOUTH EAST WEST NORTHEAST NORTHWEST SOUTHEAST SOUTHWEST NEW
COLORADO FLORIDA ARIZONA MICHIGAN OHIO PENN TEXAS CALIFORNIA YORK WASHINGTON VIRGINIA
""".split())

FORMS_WORDS = {w for f in FORMS for w in f.split()}
GENERIC = INSTITUTIONS | PLACES | FORMS_WORDS

SECTOR_WORDS = set("""
QUANTUM QUANTIQUE COMPUTING COMPUTER CALCULATING INFORMATION SYSTEMS SYSTEM NETWORK NETWORKS ELECTRONIC
ELECTRONICS ELECTRIC POWER GRID ENERGY DIGITAL INTELLIGENT SMART DATA SOFTWARE SOLUTIONS SOLUTION
INFRASTRUCTURE INSTRUMENT INSTRUMENTS DEVICES LICENSING PERMISSION INNOVATION INNOVATIONS VENTURES CAPITAL
INVESTMENT PORTFOLIO INTELLECTUAL PROPERTY FINANCIAL INDUSTRY INDUSTRIAL DEVELOPMENT TECHNOLOGY MANUFACTURING
APPLICATION APPLIED SERVICES SERVICE MACHINES EQUIPMENT AEROSPACE SPACE VISION BIOLOGICAL HEALTH MEDICINE
MEDICAL ATOMIC ALTERNATIVE MOBILE MOBILITY TELECOM TELECOMMUNICATIONS COMMUNICATION COMMUNICATIONS ENGINEERING
AGRICULTURAL MARITIME PETROLEUM MINERALS JIAOTONG EDUCATION HIGHER OPTICS OPTO PHOTONICS NANO NANOSCIENCE
NANOTECHNOLOGY TOOLS CLOUD STORAGE SECURITY DEFENCE DEFENSE MILITARY PLA ARMY NAVY FORCE MOTOR PLATFORMS
PLATFORM INTELLIGENCE ARTIFICIAL CIRCUITS INTEGRATED TIME SILICON WAVE GLOBAL STAR GREAT SUPER ADVANCED
FUTURE OPEN SECOND MATRIX BANK
CANADA ISRAEL SWITZERLAND NETHERLANDS OXFORD CAMBRIDGE TOKYO SEOUL PARIS BERLIN MUNICH MUNCHEN ZURICH
VIENNA WIEN DELFT NIPPON KR
AS BY REPRESENTED ELECT RES IND FOUND NAT SCI INT
""".split())

GENERIC_QC = GENERIC | SECTOR_WORDS

## 6. Run the pipeline (clean → brand key → fuzzy), one applicant type at a time

In [29]:
labels = pd.concat(harmonise(g["raw_name"], GENERIC_QC) + " | " + t for t, g in df.groupby("entity_type"))
df["group_id"] = make_ids(labels)
df["group_name"] = make_names(df["raw_name"], df["group_id"])

## 7. Final table
One row per family × organisation. `weight` splits each family equally among its organisations, so the weights add up to the number of families.

In [30]:
out = df.drop_duplicates(["docdb_family_id", "group_id"])[["docdb_family_id", "group_id", "group_name", "entity_type"]]
out.to_csv(OUTPUT_CSV, index=False)

print(f"{df['raw_name'].nunique():,} names -> {out['group_id'].nunique():,} organisations, {out['docdb_family_id'].nunique():,} families")
out["group_name"].value_counts().head(15)

3,107 names -> 2,162 organisations, 12,181 families


group_name
ORIGIN QUANTUM COMPUTING TECHNOLOGY (HEFEI) CO., LTD.                       904
IBM CORP                                                                    763
BEIJING BAIDU NETCOM SCIENCE AND TECHNOLOGY CO., LTD.                       354
MICROSOFT TECH LICENSING LLC                                                254
GOOGLE LLC                                                                  243
FUJITSU LTD                                                                 229
D WAVE SYSTEMS INC                                                          201
IONQ INC                                                                    173
INTEL CORP                                                                  166
RUBAN QUANTUM TECHNOLOGY CO., LTD.                                          153
QUANTINUUM LLC                                                              148
NEC CORP                                                                    130
RED HAT INC                  